# eg4 (v2) — errors

v1 checked commands with a tokenizer (syntax) and a command cache (semantics) before sending. In v2 the Construction type is the checker: the closed world of 28 heads (`UnknownHead`), the statement grammar (`ParseError`) and the per-head arity (`arity_ok`) are all decided before anything is sent; runtime errors come back from the host as the applet's reply.

In [1]:
import sys; sys.path.insert(0, '/Users/manabu/work/ggblab-replay')
from ggblab.parse import parse_statement, parse_cell, UnknownHead, ParseError
from ggblab.construction import arity_ok, render
import ggblab.host.html_host as H; H.DEPLOY = 'https://cdn.geogebra.org/apps/deployggb.js'
from ggblab import GeoGebra

## 1. closed world: a head outside the 28 is refused before sending

In [2]:
for text in ['x = Tangent(:A, :c)', 'Circle(:O, Tangent(:A, :c))', '   ', 'Circle(A, ']:
    try:
        s = parse_statement(text); print(repr(text), '->', s)
    except UnknownHead as e: print(repr(text), '-> UnknownHead', e.head)
    except ParseError as e: print(repr(text), '-> ParseError', e)

## 2. arity: decided from the signature table, not by the applet

In [3]:
for text in ['TriangleCenter(:A, :B, :C, 2)', 'TriangleCenter(:A, :B, :C)', 'Polygon(:A, :B, :C, :D, :E)']:
    s = parse_statement(text); print(text, '-> arity_ok', arity_ok(s), '| wire:', render(s))

## 3. runtime errors are the applet's reply (host verb `eval`), not an exception on the way out

In [4]:
g = GeoGebra(appName='suite', showAlgebraInput=True); g

In [5]:
print('ok  :', g.command('A = (1, 2)', 'Circle(A, 1)', timeout=60))
print('bad :', g.command('Circle(B, 1)', timeout=60))   # B undefined → the applet returns no label
print('DONE')